<a href="https://colab.research.google.com/github/alinawang1/george_spucomnist/blob/main/george_spucomnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Deep neural networks often exploit non-predictive features that are spuriously correlated with class labels, leading to poor performance on groups of examples without such features. Using the SpuCo Package ([SpuCo Documentation](https://spuco.readthedocs.io/en/latest/)), we'd like you to implement a simple method to remedy spurious correlations in [SpuCoMNIST](https://spuco.readthedocs.io/en/latest/reference/datasets.html#spuco.datasets.spuco_mnist.SpuCoMNIST) (use default parameters to initialize the dataset).

The method ([George](https://arxiv.org/abs/2011.12945)) we'd like you to implement has a 3 step pipeline:
1. Train a model using ERM
2. Cluster inputs based on the output they produce for ERM
3. Retrain using "Group-Balancing" to ensure in each batch each group appears equally.

In [ ]:
!pip install spuco

In [ ]:
import torch
from spuco.datasets import SpuCoMNIST, SpuriousFeatureDifficulty
import random
import numpy as np

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

classes = [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]
difficulty = SpuriousFeatureDifficulty.MAGNITUDE_LARGE

trainset = SpuCoMNIST(root="data/mnist/", spurious_feature_difficulty=difficulty,
                      classes=classes, spurious_correlation_strength=0.995,
                      split="train")
trainset.initialize()

valset = SpuCoMNIST(root="data/mnist/", spurious_feature_difficulty=difficulty,
                    classes=classes, split="val")
valset.initialize()

testset = SpuCoMNIST(root="data/mnist/", spurious_feature_difficulty=difficulty,
                     classes=classes, split="test")
testset.initialize()

for g, idx in trainset.group_partition.items():
    print(g, len(idx))

cuda
(2, 2) 8965
(0, 0) 10082
(4, 4) 9393
(1, 1) 9623
(3, 3) 9698
(4, 1) 11
(1, 2) 11
(4, 3) 18
(2, 1) 13
(3, 1) 14
(3, 4) 11
(2, 4) 18
(4, 2) 11
(0, 3) 15
(1, 4) 14
(0, 4) 15
(0, 2) 9
(3, 2) 12
(0, 1) 12
(1, 0) 15
(1, 3) 9
(3, 0) 12
(2, 0) 5
(4, 0) 8
(2, 3) 10


In [ ]:
from torch.optim import SGD
from spuco.models import model_factory
from spuco.robust_train import ERM
from spuco.evaluate import Evaluator

model = model_factory("lenet", trainset[0][0].shape, trainset.num_classes).to(device)

erm = ERM(
    model=model,
    trainset=trainset,
    batch_size=64,
    optimizer=SGD(model.parameters(), lr=1e-2, momentum=0.9, nesterov=True),
    num_epochs=2,
    device=device,
    verbose=False,
)
erm.train()

evaluator = Evaluator(
    testset=testset,
    group_partition=testset.group_partition,
    group_weights=trainset.group_weights,
    batch_size=64,
    model=model,
    device=device,
    verbose=True,
)
evaluator.evaluate()
print("ERM average accuracy:", evaluator.average_accuracy)
print("ERM worst-group accuracy:", evaluator.worst_group_accuracy)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Evaluating group-wise accuracy:   4%|▍    

Group (0, 0) Accuracy: 100.0


Evaluating group-wise accuracy:   8%|▊         | 2/25 [00:01<00:12,  1.77it/s]

Group (0, 1) Accuracy: 0.0


Evaluating group-wise accuracy:  12%|█▏        | 3/25 [00:01<00:13,  1.62it/s]

...

Group (4, 2) Accuracy: 0.0


Evaluating group-wise accuracy:  96%|█████████▌| 24/25 [00:13<00:00,  1.90it/s]

Group (4, 3) Accuracy: 0.0


Evaluating group-wise accuracy: 100%|██████████| 25/25 [00:13<00:00,  1.79it/s]

Group (4, 4) Accuracy: 100.0
ERM average accuracy: 99.49379218398467
ERM worst-group accuracy: ((0, 1), 0.0)


In [ ]:
from spuco.group_inference import Cluster, ClusterAlg

logits = erm.trainer.get_trainset_outputs()

cluster = Cluster(
    Z=logits,
    class_labels=trainset.labels,
    cluster_alg=ClusterAlg.KMEANS,
    num_clusters=5,
    device=device,
    verbose=True,
)
group_partition = cluster.infer_groups()

for g, idx in sorted(group_partition.items()):
    print(g, len(idx))

Clustering class-wise: 100%|██████████| 5/5 [00:00<00:00, 10.15it/s]

(0, 0) 3654
(0, 1) 36
(0, 2) 4667
(0, 3) 1761
(0, 4) 15
(1, 0) 3195
(1, 1) 1493
(1, 2) 1669
(1, 3) 3266
(1, 4) 49
(2, 0) 4937
(2, 1) 15
(2, 2) 18
(2, 3) 13
(2, 4) 4028
(3, 0) 9698
(3, 1) 14
(3, 2) 12
(3, 3) 11
(3, 4) 12
(4, 0) 7183
(4, 1) 11
(4, 2) 26
(4, 3) 11
(4, 4) 2210


In [ ]:
true_group = {}
for g, idx in trainset.group_partition.items():
    for i in idx:
        true_group[i] = g

for c, idx in sorted(group_partition.items()):
    counts = {}
    for i in idx:
        counts[true_group[i]] = counts.get(true_group[i], 0) + 1
    print(c, counts)

(0, 0) {(0, 0): 3654}
(0, 1) {(0, 4): 15, (0, 2): 9, (0, 1): 12}
(0, 2) {(0, 0): 4667}
(0, 3) {(0, 0): 1761}
(0, 4) {(0, 3): 15}
(1, 0) {(1, 1): 3195}
(1, 1) {(1, 1): 1493}
(1, 2) {(1, 1): 1669}
(1, 3) {(1, 1): 3266}
(1, 4) {(1, 2): 11, (1, 4): 14, (1, 0): 15, (1, 3): 9}
(2, 0) {(2, 2): 4937}
(2, 1) {(2, 0): 5, (2, 3): 10}
(2, 2) {(2, 4): 18}
(2, 3) {(2, 1): 13}
(2, 4) {(2, 2): 4028}
(3, 0) {(3, 3): 9698}
(3, 1) {(3, 1): 14}
(3, 2) {(3, 0): 12}
(3, 3) {(3, 4): 11}
(3, 4) {(3, 2): 12}
(4, 0) {(4, 4): 7183}
(4, 1) {(4, 2): 11}
(4, 2) {(4, 3): 18, (4, 0): 8}
(4, 3) {(4, 1): 11}
(4, 4) {(4, 4): 2210}


In [ ]:
from spuco.robust_train import GroupBalanceBatchERM

model_gb = model_factory("lenet", trainset[0][0].shape, trainset.num_classes).to(device)

gb_erm = GroupBalanceBatchERM(
    model=model_gb,
    trainset=trainset,
    group_partition=group_partition,
    batch_size=64,
    optimizer=SGD(model_gb.parameters(), lr=1e-2, momentum=0.9, nesterov=True),
    num_epochs=2,
    device=device,
    verbose=False,
)
gb_erm.train()

evaluator_gb = Evaluator(
    testset=testset,
    group_partition=testset.group_partition,
    group_weights=trainset.group_weights,
    batch_size=64,
    model=model_gb,
    device=device,
    verbose=True,
)
evaluator_gb.evaluate()
print("George average accuracy:", evaluator_gb.average_accuracy)
print("George worst-group accuracy:", evaluator_gb.worst_group_accuracy)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Evaluating group-wise accuracy:   4%|▍    

Group (0, 0) Accuracy: 100.0


Evaluating group-wise accuracy:   8%|▊         | 2/25 [00:01<00:17,  1.33it/s]

Group (0, 1) Accuracy: 76.35933806146572


Evaluating group-wise accuracy:  12%|█▏        | 3/25 [00:02<00:14,  1.51it/s]

...

Group (4, 2) Accuracy: 51.13350125944584


Evaluating group-wise accuracy:  96%|█████████▌| 24/25 [00:14<00:00,  1.44it/s]

Group (4, 3) Accuracy: 88.63636363636364


Evaluating group-wise accuracy: 100%|██████████| 25/25 [00:14<00:00,  1.70it/s]

Group (4, 4) Accuracy: 98.73737373737374
George average accuracy: 98.9954290139682
George worst-group accuracy: ((2, 4), 18.449197860962567)


In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Method": ["ERM", "George (cluster + group-balanced)"],
    "Average accuracy (%)": [evaluator.average_accuracy, evaluator_gb.average_accuracy],
    "Worst-group accuracy (%)": [evaluator.worst_group_accuracy[1], evaluator_gb.worst_group_accuracy[1]],
    "Worst group": [evaluator.worst_group_accuracy[0], evaluator_gb.worst_group_accuracy[0]],
})
results.round(2)

,Method,Average accuracy (%),Worst-group accuracy (%),Worst group
0,ERM,99.49,0.00,"(0, 1)"
1,George (cluster + group-balanced),99.00,18.45,"(2, 4)"


## Results

ERM reaches 99.5% average accuracy but 0% worst-group accuracy: 100% on every majority group and 0% on every minority group, meaning it learned background color instead of digit shape.

George raised worst-group accuracy from 0% to 18.4% and lifted most minority groups from 0% to roughly 50 - 90%, while average accuracy barely changed (99.5% --> 99.0%).

Class 3 was recovered perfectly (one majority cluster plus four pure minority clusters). In the other classes, K-means with k = 5 split the majority group into several clusters and merged some minority groups together; in class 1, all four minority groups ended up in a single cluster.

Notably, the worst group after George, (2,4), was clustered perfectly (its own pure cluster), while class 1's fully merged minority groups still reached 65 - 86%. So clustering errors were not the main bottleneck. The remaining gap is more likely due to how few examples each minority group has (~5 - 18), which limits what the model can learn even with balanced sampling.

## Limitations
1. Using k = 5 for every class caused majority groups to be split and minority groups to be merged. The original George paper selects the number of clusters automatically (via silhouette scores), which could avoid this.
2. Before fixing a random seed, results varied substantially between runs (worst-group accuracy ranged from ~18% to ~48%), largely driven by how well clustering recovered the groups. A more complete evaluation would average over several seeds.
3. Group-balanced sampling repeats the same handful of minority examples many times, which risks overfitting to them.
